In [0]:
%run ./00_config_and_helpers

In [0]:
import requests, json, time, os
from datetime import datetime, timezone

dbutils.widgets.text("github_token", "")
dbutils.widgets.text("max_pages", "3")       # pages per repo/endpoint, 0 = no limit
dbutils.widgets.text("endpoints", "commits,issues")

TOKEN     = dbutils.widgets.get("github_token").strip()
MAX_PAGES = int(dbutils.widgets.get("max_pages"))
ENDPOINTS = [e.strip() for e in dbutils.widgets.get("endpoints").split(",")]
HEADERS = {"Accept": "application/vnd.github+json"}
if TOKEN:
    HEADERS["Authorization"] = f"Bearer {TOKEN}"
else:
    print("WARNING: no token, GitHub will only allow 60 requests/hour")

PAGES_PER_FILE = 10
INGEST_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")
MAX_WAIT = 3700   # seconds we are willing to sleep for a rate limit reset

def get_page(url, params=None):
    """Returns (records, next_url). Follows GitHub's Link header (cursor style)."""
    for attempt in range(6):
        r = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if r.status_code == 200:
            return r.json(), r.links.get("next", {}).get("url")
        if r.status_code in (403, 429):
            if r.headers.get("Retry-After"):
                wait = int(r.headers["Retry-After"]) + 2
            elif r.headers.get("X-RateLimit-Remaining") == "0":
                wait = max(int(r.headers.get("X-RateLimit-Reset", time.time() + 60)) - int(time.time()), 1) + 2
            else:
                raise Exception(f"GitHub {r.status_code}: {r.text[:200]}")
            if wait > MAX_WAIT:
                raise Exception(f"Rate limit exhausted, resets in {wait}s")
            print(f"Rate limited, sleeping {wait}s")
            time.sleep(wait)
            continue
        if r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        raise Exception(f"GitHub {r.status_code}: {r.text[:200]}")
    raise Exception("Too many retries")

def ingest(repo, endpoint):
    start = datetime.now()
    repo_name = repo.split("/")[1]
    url = f"https://api.github.com/repos/{repo}/{endpoint}"
    params = {"per_page": 100, "since": f"{START_DATE}T00:00:00Z"}
    if endpoint == "commits":
        params["until"] = f"{END_DATE}T23:59:59Z"
    else:
        params["state"] = "all"
    out_dir = f"{BASE_PATH}/landing/{LOAD_TYPE}/{endpoint}/{repo_name}/{INGEST_DATE}"
    os.makedirs(out_dir, exist_ok=True)

    buffer, files, total = [], 0, 0

    def flush():
        nonlocal buffer, files
        if not buffer:
            return
        path = f"{out_dir}/{endpoint}_{BATCH_ID}_part{files:04d}.json"
        with open(path, "w") as f:
            for rec in buffer:                 # one JSON record per line
                f.write(json.dumps(rec) + "\n")
        files += 1
        buffer = []

    try:
        page = 1
        data, nxt = get_page(url, params)
        while True:
            if not data:
                break
            buffer.extend(data)
            total += len(data)
            if len(buffer) >= PAGES_PER_FILE * 100:
                flush()
            if not nxt:
                break
            if MAX_PAGES and page >= MAX_PAGES:
                break
            page += 1
            data, nxt = get_page(nxt)
        flush()
        log_run("Source-to-Raw", f"{repo}/{endpoint} -> {out_dir}", start, "Success", inserted=total)
        print(f"{repo} {endpoint}: {total} records, {files} files")
    except Exception as e:
        flush()   # keep what we already downloaded
        log_run("Source-to-Raw", f"{repo}/{endpoint}", start, "Failure", error=str(e)[:500])
        print(f"FAILED {repo} {endpoint}: {e}")

for repo in REPOS:
    for ep in ENDPOINTS:
        ingest(repo, ep)

In [0]:
display(spark.sql("SELECT * FROM pipeline_execution_logs ORDER BY start_time DESC"))
display(dbutils.fs.ls(f"{BASE_PATH}/landing"))